# YOLOX: validation calibration and held-back test

Run after notebook 08 completes. This notebook performs inference only; no model training occurs. It selects confidence on validation, saves that choice, then applies the frozen value to test. Test results must not be used to choose thresholds. Since earlier models already used these test images for development evaluation, this is not a new blind benchmark.

Correct detections require class match and IoU >=0.50. Negative false positives count any accepted box on a negative image. Counts and metrics are reported by source as well as overall. MIDV-2020 has no driving licences, so that recall is N/A.

The calibration goal is >=90% recall for both classes. Among eligible thresholds, choose lowest negative false-positive rate, then highest minimum recall, then lowest threshold. If none qualify, retain 0.25 and explicitly report calibration failure. <=5% negative false positives is also required to meet the overall target.

The phone-photo section uses only expected image-level labels; inspect predicted boxes manually because no ground-truth boxes are available.

In [1]:
from pathlib import Path
import sys
import json
import numpy as np
import torch
from IPython.display import display, HTML
import html

PROJECT = Path("/home/joe/PycharmProjects/id-capture")
SOURCE = PROJECT / "training/vendor/YOLOX"
DATA = PROJECT / "training/data/yolox_all_sources"
for directory in (str(PROJECT), str(SOURCE)):
    if directory not in sys.path:
        sys.path.insert(0, directory)

from training.yolox_workflow import (
    make_model, predict_records, summarize, audit, save_json, NAMES,
)

pointer = json.loads((DATA / "latest_run.json").read_text())
RUN = Path(pointer["run"])
manifest = json.loads(Path(pointer["manifest"]).read_text())
records = manifest["records"]
audit(records)
assert torch.cuda.is_available(), "CUDA is required."
SIZE = pointer["size"]
model = make_model(pointer["best"]).cuda().eval()

def show_table(rows):
    columns = list(rows[0])
    text = "<table><tr>" + "".join(f"<th>{html.escape(c)}</th>" for c in columns) + "</tr>"
    for row in rows:
        text += "<tr>" + "".join(
            f"<td>{html.escape('N/A' if row[c] is None else str(row[c]))}</td>" for c in columns
        ) + "</tr>"
    display(HTML(text + "</table>"))

print("Checkpoint:", pointer["best"])

train 13100 {('midv500', 'none'): 4292, ('midv500', 'driving_licence'): 2362, ('midv500', 'passport'): 2346, ('midv2019', 'none'): 1680, ('midv2019', 'driving_licence'): 960, ('midv2019', 'passport'): 960, ('midv2020', 'none'): 200, ('midv2020', 'passport'): 300}
val 4400 {('midv500', 'none'): 1511, ('midv500', 'passport'): 890, ('midv500', 'driving_licence'): 599, ('midv2019', 'none'): 600, ('midv2019', 'passport'): 360, ('midv2019', 'driving_licence'): 240, ('midv2020', 'none'): 100, ('midv2020', 'passport'): 100}
test 4500 {('midv500', 'passport'): 884, ('midv500', 'none'): 1537, ('midv500', 'driving_licence'): 579, ('midv2019', 'passport'): 360, ('midv2019', 'driving_licence'): 240, ('midv2019', 'none'): 600, ('midv2020', 'none'): 300}
PASS: no document design crosses splits; all images exist.
Loaded pretrained tensors: 462 | New class-head tensors: 0
Checkpoint: /home/joe/PycharmProjects/id-capture/training/runs/yolox_tiny_20261021_164408_054475/best.pth


## Choose threshold using validation only

In [2]:
validation = [r for r in records if r["split"] == "val"]
validation_predictions = predict_records(model, validation, SIZE)
thresholds = sorted(set([0.25] + [round(x, 2) for x in np.arange(0.10, 0.96, 0.05)]))
sweep = [summarize(validation, validation_predictions, threshold)[0]
         for threshold in thresholds]
eligible = [row for row in sweep if row["licence_recall"] >= 0.90
            and row["passport_recall"] >= 0.90]
if eligible:
    choice = min(eligible, key=lambda row: (
        row["negative_fp"],
        -min(row["licence_recall"], row["passport_recall"]),
        row["threshold"],
    ))
else:
    choice = next(row for row in sweep if row["threshold"] == 0.25)

THRESHOLD = choice["threshold"]
calibration = dict(
    checkpoint=pointer["best"], threshold=THRESHOLD,
    recall_target_met=bool(eligible),
    all_targets_met=bool(eligible) and choice["negative_fp"] <= 0.05,
    sweep=sweep,
)
save_json(RUN / "calibration.json", calibration)
show_table(sweep)
print("Selected threshold:", THRESHOLD)
print("Recall calibration passed:", calibration["recall_target_met"])
print("All validation targets met:", calibration["all_targets_met"])
show_table(summarize(validation, validation_predictions, THRESHOLD))
del validation_predictions

Inference 16/4400 | 0.4% | Elapsed 0:00:00 | Remaining 0:03:12 | Estimated finish 2026-10-21 19:07:40 BST
Inference 816/4400 | 18.5% | Elapsed 0:00:04 | Remaining 0:00:21 | Estimated finish 2026-10-21 19:04:54 BST
Inference 1616/4400 | 36.7% | Elapsed 0:00:08 | Remaining 0:00:15 | Estimated finish 2026-10-21 19:04:51 BST
Inference 2416/4400 | 54.9% | Elapsed 0:00:12 | Remaining 0:00:10 | Estimated finish 2026-10-21 19:04:50 BST
Inference 3216/4400 | 73.1% | Elapsed 0:00:16 | Remaining 0:00:05 | Estimated finish 2026-10-21 19:04:50 BST
Inference 4016/4400 | 91.3% | Elapsed 0:00:19 | Remaining 0:00:01 | Estimated finish 2026-10-21 19:04:49 BST
Inference 4400/4400 | 100.0% | Elapsed 0:00:21 | Remaining 0:00:00 | Finished 2026-10-21 19:04:49 BST


source,threshold,images,licence_count,passport_count,negative_count,licence_recall,passport_recall,negative_fp
combined,0.1,4400,839,1350,2211,0.9153754469606674,0.9881481481481481,0.1895070104025328
combined,0.15,4400,839,1350,2211,0.9082240762812872,0.9881481481481481,0.17955676164631387
combined,0.2,4400,839,1350,2211,0.902264600715137,0.9881481481481481,0.17503392130257803
combined,0.25,4400,839,1350,2211,0.8939213349225268,0.9881481481481481,0.17322478516508366
combined,0.3,4400,839,1350,2211,0.8891537544696066,0.9881481481481481,0.16870194482134782
combined,0.35,4400,839,1350,2211,0.8831942789034565,0.9881481481481481,0.16644052464947987
combined,0.4,4400,839,1350,2211,0.8784266984505363,0.9874074074074074,0.1628222523744912
combined,0.45,4400,839,1350,2211,0.8736591179976162,0.9859259259259259,0.16191768430574402
combined,0.5,4400,839,1350,2211,0.867699642431466,0.9851851851851852,0.15694255992763456
combined,0.55,4400,839,1350,2211,0.8629320619785459,0.9844444444444445,0.15377657168701944


Selected threshold: 0.2
Recall calibration passed: True
All validation targets met: False


source,threshold,images,licence_count,passport_count,negative_count,licence_recall,passport_recall,negative_fp
combined,0.2,4400,839,1350,2211,0.902264600715137,0.9881481481481481,0.17503392130257803
midv2019,0.2,1200,240,360,600,0.8541666666666666,1.0,0.255
midv2020,0.2,200,0,100,100,N/A,0.97,0.3
midv500,0.2,3000,599,890,1511,0.9215358931552587,0.9853932584269663,0.13500992720052946


## Test with the frozen validation threshold

Run this after calibration. It loads the saved threshold rather than choosing one from test results. Keep the resulting report even if performance is disappointing.

In [3]:
calibration = json.loads((RUN / "calibration.json").read_text())
assert calibration["checkpoint"] == pointer["best"]
THRESHOLD = calibration["threshold"]
test_records = [r for r in records if r["split"] == "test"]
test_predictions = predict_records(model, test_records, SIZE)
test_rows = summarize(test_records, test_predictions, THRESHOLD)
show_table(test_rows)
test_report = dict(checkpoint=pointer["best"], threshold=THRESHOLD,
                   iou=0.50, split="test", rows=test_rows,
                   limitation="Previously inspected development test; correlated video frames.")
save_json(RUN / "test_report.json", test_report)

combined = test_rows[0]
passed = (combined["licence_recall"] >= 0.90
          and combined["passport_recall"] >= 0.90
          and combined["negative_fp"] <= 0.05)
print("Combined test targets met:", passed)
print("Saved report:", RUN / "test_report.json")
del test_predictions

Inference 16/4500 | 0.4% | Elapsed 0:00:00 | Remaining 0:00:32 | Estimated finish 2026-10-21 19:06:07 BST
Inference 816/4500 | 18.1% | Elapsed 0:00:04 | Remaining 0:00:20 | Estimated finish 2026-10-21 19:06:00 BST
Inference 1616/4500 | 35.9% | Elapsed 0:00:09 | Remaining 0:00:16 | Estimated finish 2026-10-21 19:05:59 BST
Inference 2416/4500 | 53.7% | Elapsed 0:00:13 | Remaining 0:00:11 | Estimated finish 2026-10-21 19:05:59 BST
Inference 3216/4500 | 71.5% | Elapsed 0:00:17 | Remaining 0:00:06 | Estimated finish 2026-10-21 19:05:58 BST
Inference 4016/4500 | 89.2% | Elapsed 0:00:21 | Remaining 0:00:02 | Estimated finish 2026-10-21 19:05:58 BST
Inference 4500/4500 | 100.0% | Elapsed 0:00:23 | Remaining 0:00:00 | Finished 2026-10-21 19:05:58 BST


source,threshold,images,licence_count,passport_count,negative_count,licence_recall,passport_recall,negative_fp
combined,0.2,4500,819,1244,2437,0.945054945054945,0.9919614147909968,0.08001641362330734
midv2019,0.2,1200,240,360,600,0.9708333333333333,1.0,0.06833333333333333
midv2020,0.2,300,0,0,300,N/A,N/A,0.03333333333333333
midv500,0.2,3000,579,884,1537,0.9343696027633851,0.9886877828054299,0.0936890045543266


Combined test targets met: False
Saved report: /home/joe/PycharmProjects/id-capture/training/runs/yolox_tiny_20261021_164408_054475/test_report.json


## Separate uploaded-phone sanity check

These photos were excluded from training and threshold selection. This cell uses EXIF-corrected photos in memory and leaves originals untouched. Type matches alone do not establish correct localisation.

In [ ]:
from PIL import Image, ImageOps
import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from training.yolox_workflow import letterbox
from yolox.utils import postprocess

phone_rows = []
extensions = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".webp"}
for expected in (*NAMES, "none"):
    folder = PROJECT / "training/data/raw" / expected
    if not folder.is_dir():
        continue
    for path in sorted(folder.iterdir()):
        if path.suffix.lower() not in extensions:
            continue
        with Image.open(path) as original:
            rgb = np.array(ImageOps.exif_transpose(original).convert("RGB"))
        image, ratio = letterbox(np.ascontiguousarray(rgb[:, :, ::-1]), SIZE)
        with torch.inference_mode():
            result = postprocess(model(torch.from_numpy(image).unsqueeze(0).cuda()),
                                 2, THRESHOLD, 0.65)[0]
        predicted, confidence = "none", None
        fig, ax = plt.subplots(figsize=(7, 6))
        ax.imshow(rgb)
        if result is not None:
            result = result.cpu().numpy()
            scores = result[:, 4] * result[:, 5]
            top = int(scores.argmax())
            predicted = NAMES[int(result[top, 6])]
            confidence = float(scores[top])
            for detection in result:
                x1, y1, x2, y2 = detection[:4] / ratio
                ax.add_patch(Rectangle((x1, y1), x2-x1, y2-y1,
                                       fill=False, edgecolor="lime", linewidth=2))
        phone_rows.append(dict(photo=path.name, expected=expected, predicted=predicted,
                               confidence=confidence, type_match=expected == predicted))
        ax.set_title(f"{path.name}: expected {expected}, predicted {predicted}")
        ax.axis("off")
        plt.show()
        plt.close(fig)
if phone_rows:
    show_table(phone_rows)
    save_json(RUN / "phone_sanity_check.json", phone_rows)
else:
    print("No phone photos found.")